# **Block 1 — Imports**

We bring in the tools required for the exact workflow you will repeat in real projects: load data, split into train and test, scale features so distance is fair, train a kNN classifier, and evaluate predictions with a confusion matrix and a few core metrics.

In [1]:
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score


# **Block 2 — Inputs (X = features, y = labels)**

kNN is “learning by similarity,” which means the model makes predictions by comparing a new point to examples it has already seen. That only works if we clearly separate our inputs, which are the features the model uses to compare points, from our labels, which are the correct answers the model is trying to predict. We use a built-in dataset here so you can focus on the kNN workflow instead of spending time on data cleaning, because the goal is to recognize the structure you’ll reuse later.

In [2]:
# Load a clean binary classification dataset
data = load_breast_cancer()

# X = features (inputs). Shape: (rows, features)
X = data.data

# y = labels (targets). Shape: (rows,)
y = data.target

# Quick sanity checks
print("X shape:", X.shape)
print("y shape:", y.shape)
print("First 10 labels:", y[:10])

# Quick class balance check (useful context for interpreting metrics)
unique, counts = np.unique(y, return_counts=True)
print("Class counts:", dict(zip(unique, counts)))


X shape: (569, 30)
y shape: (569,)
First 10 labels: [0 0 0 0 0 0 0 0 0 0]
Class counts: {np.int64(0): np.int64(212), np.int64(1): np.int64(357)}


# **Block 3 — Train/Test split**

We split the dataset so we can measure whether the model generalizes to new data. Training performance can look great even when a model is not actually learning something that transfers, so the test set is where you get the honest result. We also use stratification to keep the proportion of classes similar in both splits, which helps prevent misleading evaluations caused by an unlucky split.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,      # 25% test, 75% train
    random_state=42,     # reproducible split
    stratify=y           # keep class balance similar in train and test
)

print("Train rows:", X_train.shape[0])
print("Test rows :", X_test.shape[0])


Train rows: 426
Test rows : 143


# **Block 4 — Feature scaling (the kNN “make it fair” step)**

This is the most important preparation step for kNN. Because kNN is distance-based, features with large numeric ranges can dominate the distance calculation, which can make the model behave as if smaller-scale features barely exist. Scaling fixes this by putting features on comparable scales. The critical rule is that we fit the scaler on the training set only, then apply that same transformation to the test set, so we don’t accidentally leak test information.

In [4]:
# Create a scaler
scaler = StandardScaler()

# Fit the scaler ONLY on training data, then transform training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using the SAME scaler (do not fit again)
X_test_scaled = scaler.transform(X_test)

print("Scaled train shape:", X_train_scaled.shape)
print("Scaled test shape :", X_test_scaled.shape)


Scaled train shape: (426, 30)
Scaled test shape : (143, 30)


# **Block 5 — Train kNN (choose k)**

kNN has one main “knob” that controls how it behaves: k, the number of neighbors we look at. With a small k, the model is very sensitive to local noise and outliers, because a few points can strongly influence the prediction. With a larger k, the model becomes smoother and more stable, because it averages over more nearby points. We start with a simple, common baseline like k = 5.

In [5]:
k = 5

# Create the kNN classifier
# n_neighbors = how many nearby points we consult before deciding
model = KNeighborsClassifier(n_neighbors=k)

# Fit the model
# For kNN, "training" mainly means storing the training data so it can be compared later
model.fit(X_train_scaled, y_train)

print("kNN trained with k =", k)


kNN trained with k = 5


# **Block 6 — Predict on test data (what the model decides)**

Once the model is trained, we ask it to predict labels for the test set. This is the moment where “learning by similarity” becomes real: each test point gets compared to the training points, and the model decides the label based on the nearest neighbors.

In [6]:
# Predict class labels for the test set
y_pred = model.predict(X_test_scaled)

# Quick peek at the first few predictions
print("First 15 predictions:", y_pred[:15])
print("First 15 true labels:", y_test[:15])


First 15 predictions: [1 0 1 1 0 0 0 0 0 1 1 1 0 0 1]
First 15 true labels: [1 0 1 1 0 0 0 0 0 1 1 1 0 0 1]


# **Block 7 — Evaluate (confusion matrix + accuracy/precision/recall)**

Evaluation is where you stop guessing and start measuring. The confusion matrix tells you what kinds of mistakes the model is making, and the metrics summarize performance from different angles. Accuracy gives overall correctness, precision tells you how trustworthy positive predictions are, and recall tells you how many real positives you successfully catch.

In [7]:
# Confusion matrix: [[TN, FP],
#                    [FN, TP]]
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)

# Core metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred)

print("Accuracy :", round(acc, 3))
print("Precision:", round(prec, 3))
print("Recall   :", round(rec, 3))


Confusion matrix:
 [[50  3]
 [ 0 90]]
Accuracy : 0.979
Precision: 0.968
Recall   : 1.0


# **Block 8 — Show the effect of k (small vs large, minimal code)**

This block exists for one reason: to make the “k controls smoothness” idea feel concrete. We keep everything else the same and only change k, so you can see how predictions and performance shift when the model listens to very few neighbors versus many neighbors.

In [8]:
# Compare two k values (same data, same scaling, only k changes)
k_small = 1
k_large = 15

model_small = KNeighborsClassifier(n_neighbors=k_small)
model_large = KNeighborsClassifier(n_neighbors=k_large)

model_small.fit(X_train_scaled, y_train)
model_large.fit(X_train_scaled, y_train)

pred_small = model_small.predict(X_test_scaled)
pred_large = model_large.predict(X_test_scaled)

acc_small = accuracy_score(y_test, pred_small)
acc_large = accuracy_score(y_test, pred_large)

print("Accuracy with k =", k_small, ":", round(acc_small, 3))
print("Accuracy with k =", k_large, ":", round(acc_large, 3))


Accuracy with k = 1 : 0.951
Accuracy with k = 15 : 0.958


# **Block 9 — Show why scaling matters (same k, with vs without scaling)**

This final block makes the scaling lesson undeniable. We hold k constant and compare performance when kNN uses raw features versus scaled features. Even if the accuracy difference is small on this dataset, the point is that scaling is a required habit for distance-based models, because without it you are letting measurement units decide what “similar” means.

In [9]:
k = 5

# kNN WITHOUT scaling
model_raw = KNeighborsClassifier(n_neighbors=k)
model_raw.fit(X_train, y_train)
pred_raw = model_raw.predict(X_test)
acc_raw = accuracy_score(y_test, pred_raw)

# kNN WITH scaling
model_scaled = KNeighborsClassifier(n_neighbors=k)
model_scaled.fit(X_train_scaled, y_train)
pred_scaled = model_scaled.predict(X_test_scaled)
acc_scaled = accuracy_score(y_test, pred_scaled)

print("Same k =", k, "but different scaling:")
print("Accuracy WITHOUT scaling:", round(acc_raw, 3))
print("Accuracy WITH scaling   :", round(acc_scaled, 3))


Same k = 5 but different scaling:
Accuracy WITHOUT scaling: 0.93
Accuracy WITH scaling   : 0.979
